# Raman intensity with the modified Raman tensor

$$
R=\begin{pmatrix} a & A e^{i\delta}\\ -A e^{i\delta} & a e^{i\phi}\end{pmatrix}
$$

**What changed compared with the reference notebooks.** The old model tensor was $\begin{pmatrix}a&Ae^{i\delta}\\-Ae^{i\delta}&a\end{pmatrix}$. The only change is the $yy$ element, $a\to a\,e^{i\phi}$, so there is a new phase parameter $\phi$ (the phase between $R_{xx}$ and $R_{yy}$). For $\phi=0$ the old tensor is recovered.

**Optical chain** (same as `Raman_intensity_ver1_rebuilt.ipynb`; the setup of `version1_Yen-san.ipynb` is reproduced as a cross-check in the Appendix):

$$
\hat x \xrightarrow{\ Q\ } E_{\rm in}\propto\sigma_+ \xrightarrow{\ R\ } E_{\rm sc} \xrightarrow{\ Q_{\rm back}=Q^T\ } E_{\rm out} \xrightarrow{\ \text{analyzer }\alpha\ } A(\alpha)\ \Rightarrow\ I(\alpha)=|A(\alpha)|^2
$$

**Conventions**
1. $\sigma_\pm=(1,\pm i)^T/\sqrt2$ in the fixed lab $xy$ basis.
2. The same quarter-wave plate is used for the incident and the returning light. For a reciprocal element the reverse-direction Jones matrix is $Q^T$ (and $Q$ is symmetric, so $Q_{\rm back}=Q$).
3. The analyzer angle is defined physically: $\alpha=0$ is the linear polarization that scattered $\sigma_+$ becomes after the plate, and $\alpha=90^\circ$ the one that scattered $\sigma_-$ becomes. So $\alpha=0$ detects $\sigma_+\to\sigma_+$ and $\alpha=90^\circ$ detects $\sigma_+\to\sigma_-$.
4. $a,\ A,\ \delta,\ \phi$ are real parameters (all complex character of the tensor comes from the phases).

**Cell map:** 1 imports · 2 symbols · 3 $\sigma_\pm$ · 4 QWP · 5 incident light · 6 Raman tensor and circular elements · 7 scattering · 8 return QWP and reference axis · 9 analyzer · 10 amplitude · 11 intensity (step by step) · 12 checks · 13 plots · Appendix (Yen-san convention).

## Cell 1 — Load libraries
**Role:** import SymPy (exact symbolic algebra), NumPy (numerical arrays) and Matplotlib (plots). `sp.init_printing()` makes SymPy output render as typeset math in Jupyter. Nothing physical is defined here.

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Math
sp.init_printing()

def show(name, expr):
    # helper: display  name = expr  as typeset LaTeX
    display(Math(name + " = " + sp.latex(expr)))

## Cell 2 — Symbols
**Role:** declare every symbol once.
* `alpha` — analyzer angle (real), measured from the reference axis defined in Cell 8.
* `a, A` — real amplitudes of the tensor; `delta` — phase of the off-diagonal element; **`phi` — new phase of the $yy$ element**.
* `Rxx, Rxy, Ryx, Ryy` — the four elements of a completely general complex tensor (used to derive general formulas first, then specialised).
* `theta_Q` — plate angle $-45^\circ$, chosen so that the plate converts $x$-polarized light into $\sigma_+$ (verified in Cell 5).
* `zero(M)` — helper that tests whether a matrix/vector is exactly zero.

Declaring `real=True` lets SymPy simplify $|e^{i\delta}|=1$ and similar expressions automatically.

In [ ]:
alpha = sp.symbols('alpha', real=True)
a, A, delta, phi = sp.symbols('a A delta phi', real=True)
Rxx, Rxy, Ryx, Ryy = sp.symbols('R_xx R_xy R_yx R_yy', complex=True)
theta_Q = -sp.pi/4

def zero(M):
    M = sp.Matrix(M)
    return all(sp.simplify(sp.expand(z)) == 0 for z in M)

## Cell 3 — Circular basis $\sigma_\pm$
**Role:** define the lab basis and the circular basis

$$\hat x=\begin{pmatrix}1\\0\end{pmatrix},\quad \hat y=\begin{pmatrix}0\\1\end{pmatrix},\quad
\sigma_\pm=\frac1{\sqrt2}\begin{pmatrix}1\\ \pm i\end{pmatrix}$$

and check orthonormality: $\langle\sigma_+|\sigma_+\rangle=\langle\sigma_-|\sigma_-\rangle=1,\ \langle\sigma_+|\sigma_-\rangle=0$. (`.H` is the conjugate transpose.)

In [ ]:
x_hat = sp.Matrix([1, 0])
y_hat = sp.Matrix([0, 1])
sig_p = sp.Matrix([1,  sp.I]) / sp.sqrt(2)
sig_m = sp.Matrix([1, -sp.I]) / sp.sqrt(2)

print("<s+|s+> =", sp.simplify((sig_p.H*sig_p)[0]),
      "  <s-|s-> =", sp.simplify((sig_m.H*sig_m)[0]),
      "  <s+|s-> =", sp.simplify((sig_p.H*sig_m)[0]))

## Cell 4 — Jones matrix of the quarter-wave plate
**Role:** a wave plate with axis angle $\theta$ and retardation $\Gamma$ is

$$W(\theta)=U(\theta)\,\mathrm{diag}(1,e^{i\Gamma})\,U(\theta)^T,\qquad U(\theta)=\begin{pmatrix}\cos\theta&-\sin\theta\\ \sin\theta&\cos\theta\end{pmatrix}.$$

For a QWP, $\Gamma=\pi/2$, and $\theta=-45^\circ$ gives

$$Q=\frac12\begin{pmatrix}1+i & -(1-i)\\ -(1-i) & 1+i\end{pmatrix}.$$

The two properties that justify using the same matrix for the return pass are checked: $Q$ is **unitary** (lossless) and **symmetric** ($Q^T=Q$).

In [ ]:
def rot(t):
    return sp.Matrix([[sp.cos(t), -sp.sin(t)], [sp.sin(t), sp.cos(t)]])

def waveplate(theta, Gamma):
    return sp.simplify(rot(theta) * sp.diag(1, sp.exp(sp.I*Gamma)) * rot(theta).T)

Q = waveplate(theta_Q, sp.pi/2)
show("Q", Q)
print("unitary   (Q^dagger Q = 1):", zero(Q.H*Q - sp.eye(2)))
print("symmetric (Q^T = Q)       :", zero(Q.T - Q))

## Cell 5 — Incident light
**Role:** pass $x$-polarized light through the plate: $E_{\rm in}=Q\hat x$.

$$E_{\rm in}=\frac12\begin{pmatrix}1+i\\ -(1-i)\end{pmatrix}=\frac{1+i}{2}\begin{pmatrix}1\\ i\end{pmatrix}=e^{i\pi/4}\sigma_+ .$$

The global phase $e^{i\pi/4}$ does not affect any intensity. The check confirms that $\theta_Q=-45^\circ$ really produces $\sigma_+$.

In [ ]:
e4 = (1 + sp.I)/sp.sqrt(2)           # e^{i pi/4}
E_in = sp.simplify(Q * x_hat)
show("E_{in}", E_in)
print("E_in = e^{i pi/4} * sigma+ :", zero(E_in - e4*sig_p))

## Cell 6 — The new Raman tensor and its circular-basis elements
**Role:** define the tensors and extract the amplitudes in the circular basis,

$$R_{s's}=\sigma_{s'}^\dagger R\,\sigma_s,\qquad R\sigma_+=R_{++}\sigma_++R_{-+}\sigma_- .$$

$R_{++}$ is the $\sigma_+\to\sigma_+$ amplitude, $R_{-+}$ the $\sigma_+\to\sigma_-$ amplitude.

**Step-by-step by hand.** For a general tensor,

$$R\begin{pmatrix}1\\ i\end{pmatrix}=\begin{pmatrix}R_{xx}+iR_{xy}\\ R_{yx}+iR_{yy}\end{pmatrix}
\ \Rightarrow\
R_{++}=\tfrac12\big[R_{xx}+R_{yy}+i(R_{xy}-R_{yx})\big],\quad
R_{-+}=\tfrac12\big[R_{xx}-R_{yy}+i(R_{xy}+R_{yx})\big].$$

Insert $R_{xx}=a,\ R_{yy}=ae^{i\phi},\ R_{xy}=Ae^{i\delta},\ R_{yx}=-Ae^{i\delta}$ (so $R_{xy}-R_{yx}=2Ae^{i\delta}$ and $R_{xy}+R_{yx}=0$):

$$\boxed{R_{++}=\frac{a(1+e^{i\phi})}{2}+iAe^{i\delta}},\qquad
\boxed{R_{-+}=\frac{a(1-e^{i\phi})}{2}}$$

Using $1\pm e^{i\phi}$ with half angles:

$$R_{++}=a\,e^{i\phi/2}\cos\tfrac\phi2+iAe^{i\delta},\qquad R_{-+}=-i\,a\,e^{i\phi/2}\sin\tfrac\phi2 .$$

**Physical meaning:** with $\phi\neq0$ the tensor is no longer purely "$\sigma_+\to\sigma_+$": a $\sigma_+\to\sigma_-$ channel $R_{-+}\propto a(1-e^{i\phi})$ appears. It vanishes only for $\phi=0$.

The code computes $R_{\pm\pm}$ symbolically and compares them with the boxed formulas.

In [ ]:
R_gen   = sp.Matrix([[Rxx, Rxy], [Ryx, Ryy]])
R_model = sp.Matrix([[a,                       A*sp.exp(sp.I*delta)],
                     [-A*sp.exp(sp.I*delta),   a*sp.exp(sp.I*phi)]])
show("R", R_model)

def circ(R):
    # returns (R_{++}, R_{-+}) = (<s+|R|s+>, <s-|R|s+>)
    return sp.simplify((sig_p.H*R*sig_p)[0]), sp.simplify((sig_m.H*R*sig_p)[0])

# general tensor
Rpp_g, Rmp_g = circ(R_gen)
show("R_{++}^{general}", sp.expand(Rpp_g))
show("R_{-+}^{general}", sp.expand(Rmp_g))

# new model tensor
Rpp_m, Rmp_m = circ(R_model)
Rpp_hand = a*(1 + sp.exp(sp.I*phi))/2 + sp.I*A*sp.exp(sp.I*delta)
Rmp_hand = a*(1 - sp.exp(sp.I*phi))/2
show("R_{++}", sp.simplify(Rpp_m))
show("R_{-+}", sp.simplify(Rmp_m))
print("R++ matches hand formula:", sp.simplify(sp.expand(Rpp_m - Rpp_hand)) == 0)
print("R-+ matches hand formula:", sp.simplify(sp.expand(Rmp_m - Rmp_hand)) == 0)

## Cell 7 — Raman scattering $E_{\rm sc}=R\,E_{\rm in}$
**Role:** apply the tensor to the incident field and show the result component by component.

$$E_{\rm sc}=R\,E_{\rm in}=e^{i\pi/4}\,(R_{++}\sigma_++R_{-+}\sigma_-)$$

For the old tensor ($\phi=0$) this was simply $C_+E_{\rm in}$. Now a $\sigma_-$ component is generated. The check verifies the decomposition for the new tensor.

In [ ]:
E_sc = sp.simplify(R_model * E_in)
show("E_{sc}", E_sc)

decomp = e4*(Rpp_hand*sig_p + Rmp_hand*sig_m)
print("E_sc = e^{i pi/4}(R++ s+ + R-+ s-):", zero(E_sc - decomp))

## Cell 8 — Return through the same plate and the reference axis
**Role:** send the scattered light back through the plate with $Q_{\rm back}=Q^T=Q$, and define the analyzer reference axis from the physics:

* $Q\sigma_+=p_1\,\hat e_{\rm ref}$ → $\alpha=0$ axis (a real unit vector),
* $\hat e_\perp=\hat e_{\rm ref}$ rotated by $+90^\circ$ → $\alpha=90^\circ$ axis,
* $Q\sigma_-=p_2\,\hat e_\perp$.

Result (by hand): $Q\sigma_+=e^{i3\pi/4}\,\hat y$ and $Q\sigma_-=e^{i\pi/4}\,\hat x$, so $\hat e_{\rm ref}=\hat y$, $\hat e_\perp=-\hat x$, $p_1=e^{i3\pi/4}$, $p_2=-e^{i\pi/4}$, and

$$\frac{p_2}{p_1}=i .$$

This phase ratio is what produces the interference (cross) term in the intensity.

In [ ]:
Q_back = Q.T
print("Q_back == Q :", zero(Q_back - Q))

v_p = sp.simplify(Q_back * sig_p)       # scattered sigma+ after the plate
v_m = sp.simplify(Q_back * sig_m)       # scattered sigma- after the plate

k  = int(np.argmax([abs(complex(sp.N(z))) for z in v_p]))        # largest component
p1 = sp.simplify(v_p[k] / sp.sqrt(sp.simplify(v_p[k]*sp.conjugate(v_p[k]))))   # global phase of v_p
e_ref  = sp.simplify(v_p / p1)                                    # alpha = 0 axis
e_perp = sp.Matrix([[0, -1], [1, 0]]) * e_ref                     # +90 deg from e_ref
p2 = sp.simplify((e_perp.T * v_m)[0])

show("\\hat e_{ref}", e_ref)
show("\\hat e_{\\perp}", e_perp)
show("p_1", p1); show("p_2", p2)
print("Q s+ = p1 e_ref  :", zero(v_p - p1*e_ref))
print("Q s- = p2 e_perp :", zero(v_m - p2*e_perp))
print("p2/p1 =", sp.simplify(p2/p1))

## Cell 9 — Analyzer axis
**Role:** the analyzer transmits the linear polarization

$$\hat e(\alpha)=\cos\alpha\,\hat e_{\rm ref}+\sin\alpha\,\hat e_\perp$$

(here $\hat e(\alpha)=(-\sin\alpha,\ \cos\alpha)^T$). Check: at $\alpha=0$ it passes scattered $\sigma_+$ fully and blocks $\sigma_-$; at $\alpha=90^\circ$ the opposite.

In [ ]:
e_alpha = sp.cos(alpha)*e_ref + sp.sin(alpha)*e_perp
show("\\hat e(\\alpha)", e_alpha)

def transmitted(v, ang):
    return sp.simplify(sp.Abs((e_alpha.subs(alpha, ang).T * v)[0])**2)

print("alpha=0  : |t(s+)|^2 =", transmitted(v_p, 0),       "  |t(s-)|^2 =", transmitted(v_m, 0))
print("alpha=90 : |t(s+)|^2 =", transmitted(v_p, sp.pi/2), "  |t(s-)|^2 =", transmitted(v_m, sp.pi/2))

## Cell 10 — Detected amplitude $A(\alpha)$
**Role:** build the amplitude in three explicit steps.

1. Field after Raman scattering: $E_{\rm sc}=R\,E_{\rm in}$ (Cell 7).
2. Field after the returning QWP: $E_{\rm out}=Q_{\rm back}E_{\rm sc}$.
3. Projection on the analyzer: $A(\alpha)=\hat e(\alpha)^T E_{\rm out}$.

Using the circular decomposition,

$$A(\alpha)=e^{i\pi/4}\Big[p_1R_{++}\cos\alpha+p_2R_{-+}\sin\alpha\Big]
=e^{i\pi/4}p_1\Big[R_{++}\cos\alpha+i\,R_{-+}\sin\alpha\Big].$$

The code evaluates the full matrix chain and compares it with this closed form.

In [ ]:
E_out = sp.simplify(Q_back * E_sc)                 # step 2
show("E_{out}", E_out)

A_chain  = sp.expand((e_alpha.T * E_out)[0])       # step 3
A_closed = e4*(p1*Rpp_hand*sp.cos(alpha) + p2*Rmp_hand*sp.sin(alpha))
print("matrix chain == closed form:", sp.simplify(sp.expand(A_chain - A_closed)) == 0)
show("A(\\alpha)", sp.simplify(A_chain))

## Cell 11 — Intensity $I(\alpha)=|A(\alpha)|^2$, step by step
The detector measures $|A|^2$; the global phase $e^{i\pi/4}p_1$ drops out ($|e^{i\pi/4}p_1|=1$), so

$$I(\alpha)=\big|R_{++}\cos\alpha+iR_{-+}\sin\alpha\big|^2
=|R_{++}|^2\cos^2\alpha+|R_{-+}|^2\sin^2\alpha+2\,\mathrm{Im}\!\big(R_{++}R_{-+}^{*}\big)\sin\alpha\cos\alpha .$$

**Evaluate the three pieces for the new tensor** (by hand, with $R_{++}=ae^{i\phi/2}\cos\frac\phi2+iAe^{i\delta}$, $R_{-+}=-iae^{i\phi/2}\sin\frac\phi2$):

**(i)** $|R_{-+}|^2=a^2\sin^2\dfrac\phi2$.

**(ii)** $|R_{++}|^2=a^2\cos^2\frac\phi2+A^2+2aA\cos\frac\phi2\;\mathrm{Re}\big[e^{i\phi/2}(-i)e^{-i\delta}\big]$, and $\mathrm{Re}[-ie^{i(\phi/2-\delta)}]=\sin(\frac\phi2-\delta)$, hence
$$|R_{++}|^2=a^2\cos^2\tfrac\phi2+A^2+2aA\cos\tfrac\phi2\,\sin\!\big(\tfrac\phi2-\delta\big).$$

**(iii)** $R_{-+}^*=i\,a\,e^{-i\phi/2}\sin\frac\phi2$, so
$R_{++}R_{-+}^*=i\,a^2\sin\frac\phi2\cos\frac\phi2-aA\sin\frac\phi2\,e^{i(\delta-\phi/2)}$ and
$$\mathrm{Im}(R_{++}R_{-+}^*)=\tfrac{a^2}{2}\sin\phi+aA\sin\tfrac\phi2\,\sin\!\big(\tfrac\phi2-\delta\big).$$

**Final result**

$$\boxed{\begin{aligned}
I(\alpha)&=\Big[a^2\cos^2\tfrac\phi2+A^2+2aA\cos\tfrac\phi2\sin\!\big(\tfrac\phi2-\delta\big)\Big]\cos^2\alpha
+a^2\sin^2\tfrac\phi2\,\sin^2\alpha\\
&\quad+\Big[a^2\sin\phi+2aA\sin\tfrac\phi2\sin\!\big(\tfrac\phi2-\delta\big)\Big]\sin\alpha\cos\alpha
\end{aligned}}$$

Equivalent Fourier form: $I=I_0+I_c\cos2\alpha+I_s\sin2\alpha$ with $I_0=\frac12(|R_{++}|^2+|R_{-+}|^2)$, $I_c=\frac12(|R_{++}|^2-|R_{-+}|^2)$, $I_s=\mathrm{Im}(R_{++}R_{-+}^*)$.

The code below (a) computes $|A|^2$ by brute force, (b) builds the boxed formula, (c) proves they are identical (symbolically and numerically).

In [ ]:
# (a) brute force |A|^2
I_brute = sp.simplify(sp.expand_complex(sp.expand(A_chain*sp.conjugate(A_chain))))
print("Brute-force intensity:")
show("I(\\alpha)", sp.trigsimp(I_brute))

# (b) hand-derived pieces and final formula
I_pp    = a**2*sp.cos(phi/2)**2 + A**2 + 2*a*A*sp.cos(phi/2)*sp.sin(phi/2 - delta)   # |R++|^2
I_mp    = a**2*sp.sin(phi/2)**2                                                      # |R-+|^2
I_cross = a**2*sp.sin(phi)/2 + a*A*sp.sin(phi/2)*sp.sin(phi/2 - delta)               # Im(R++ R-+^*)

I_final = I_pp*sp.cos(alpha)**2 + I_mp*sp.sin(alpha)**2 + 2*I_cross*sp.sin(alpha)*sp.cos(alpha)
print("\nHand-derived intensity:")
show("I(\\alpha)", I_final)

# (c) comparison
print("\nsymbolic difference simplifies to 0 :", sp.simplify(sp.expand((I_brute - I_final).rewrite(sp.exp))) == 0)
rng = np.random.default_rng(1)
for _ in range(4):
    v = {a: rng.uniform(.2, 2), A: rng.uniform(.2, 2), delta: rng.uniform(0, 2*np.pi),
         phi: rng.uniform(0, 2*np.pi), alpha: rng.uniform(0, 2*np.pi)}
    print("brute = %.10f   formula = %.10f" % (float(I_brute.subs(v)), float(I_final.subs(v))))

# Fourier coefficients
I0 = sp.simplify((I_pp + I_mp)/2); Ic = sp.simplify((I_pp - I_mp)/2); Is = sp.simplify(I_cross)
print("\nFourier form  I = I0 + Ic cos(2 alpha) + Is sin(2 alpha):")
show("I_0", I0); show("I_c", Ic); show("I_s", Is)
print("Fourier form equals I_final:",
      sp.simplify(I0 + Ic*sp.cos(2*alpha) + Is*sp.sin(2*alpha) - I_final) == 0)

## Cell 12 — Consistency checks (special cases)
**Role:** verify that the general result reduces to known limits.

* $\phi=0$: tensor becomes the old one, $R_{-+}=0$, $I(\alpha)=|C_+|^2\cos^2\alpha$ with $C_+=a+iAe^{i\delta}$, i.e. $|C_+|^2=a^2+A^2-2aA\sin\delta$.
* $A=0$ (diagonal tensor $\mathrm{diag}(a,ae^{i\phi})$): $I=a^2\big[\cos^2\frac\phi2\cos^2\alpha+\sin^2\frac\phi2\sin^2\alpha+\sin\phi\sin\alpha\cos\alpha\big]$, which equals $a^2\cos^2(\alpha-\frac\phi2)$ — a pure $\cos^2$ shifted by $\phi/2$.
* $\phi=\pi$ with $A=0$: $R=a\,\mathrm{diag}(1,-1)$, pure $\sigma_+\to\sigma_-$, $I=a^2\sin^2\alpha$.

In [ ]:
# phi = 0  -> old tensor
Cp_abs2 = a**2 + A**2 - 2*a*A*sp.sin(delta)
print("phi=0 :  I - |C+|^2 cos^2(alpha) =",
      sp.simplify(I_final.subs(phi, 0) - Cp_abs2*sp.cos(alpha)**2))

# A = 0  -> diag(a, a e^{i phi})
I_diag = sp.simplify(I_final.subs(A, 0))
print("A=0   :  I - a^2 cos^2(alpha - phi/2) =",
      sp.simplify(sp.expand_trig(I_diag - a**2*sp.cos(alpha - phi/2)**2)))

# phi = pi, A = 0 -> a diag(1,-1)
print("phi=pi, A=0 :  I =", sp.simplify(I_final.subs({phi: sp.pi, A: 0})))

## Cell 13 — Plots of $I(\alpha)$
**Role:** convert the symbolic result into a fast NumPy function (`lambdify`) and plot $I(\alpha)$ for $\alpha=0$–$360^\circ$ (Cartesian left, polar right). The curve has $180^\circ$ period.

* **Figure 1:** vary $\phi$ at fixed $a=1,\ A=0.4,\ \delta=\pi/3$.
* **Figure 2:** vary $\delta$ at fixed $a=1,\ A=0.4,\ \phi=\pi/3$.

Intensities are in units of $a^2$ (absolute, not normalised, so changes in height are visible).

In [ ]:
I_func = sp.lambdify((alpha, a, A, delta, phi), I_final, 'numpy')
ang = np.deg2rad(np.linspace(0, 360, 721))

def make_plot(param_name, values, fixed, title):
    fig = plt.figure(figsize=(12, 4.8))
    ax1 = fig.add_subplot(121)
    ax2 = fig.add_subplot(122, projection='polar')
    for v in values:
        kw = dict(fixed); kw[param_name] = v
        I = I_func(ang, kw['a'], kw['A'], kw['delta'], kw['phi'])
        sym = r"\delta" if param_name == 'delta' else r"\phi"
        lab = "$" + sym + "=" + f"{np.rad2deg(v):.0f}" + r"^\circ$"
        ax1.plot(np.rad2deg(ang), I, label=lab)
        ax2.plot(ang, I, label=lab)
    ax1.set_xlabel(r"analyzer angle $\alpha$ (deg)   [0 = scattered $\sigma_+$ axis]")
    ax1.set_ylabel(r"$I(\alpha)$  (units of $a^2$)")
    ax1.set_xlim(0, 360); ax1.set_xticks(range(0, 361, 90)); ax1.grid(True); ax1.legend(fontsize=8)
    ax2.set_title(r"polar: $I(\alpha)$")
    fig.suptitle(title); plt.tight_layout(); plt.show()

make_plot('phi', np.deg2rad([0, 45, 90, 135, 180]),
          dict(a=1, A=0.4, delta=np.pi/3, phi=0),
          r"Effect of $\phi$  ($a=1,\ A=0.4,\ \delta=60^\circ$)")

make_plot('delta', np.deg2rad([0, 60, 120, 180, 270]),
          dict(a=1, A=0.4, delta=0, phi=np.pi/3),
          r"Effect of $\delta$  ($a=1,\ A=0.4,\ \phi=60^\circ$)")

## Appendix — Cross-check with the setup of `version1_Yen-san.ipynb`
The first reference notebook uses a slightly different optical convention:

* forward plate $Q(\pi/4)=U^T(\pi/4)\,Q_0\,U(\pi/4)$ with $Q_0=\mathrm{diag}(1,i)$ — numerically this is the **same** matrix $Q$ as above;
* a **different** matrix for the backward plate, $Q(-\pi/4)$;
* the analyzer angle $\alpha$ is measured directly from the **lab $x$ axis**: $\hat e_\alpha=(\cos\alpha,\sin\alpha)^T$.

Because the backward plate and the $\alpha$ origin differ, the formula for $I(\alpha)$ differs. The cells below apply the same new tensor to that chain and show that the result is exactly the mirror image of the main result:

$$I_{\rm Yen}(\alpha)=I(-\alpha),$$

i.e. the same curve with the sense of analyzer rotation reversed (equivalent to $\delta\to-\delta,\ \phi\to-\phi$ together with a shift). Which convention applies depends on the actual experiment (direction of the returning beam and meaning of $\alpha=0$ and of its positive sense).

In [ ]:
# Yen-san convention
def U(t):  return sp.Matrix([[sp.cos(t), -sp.sin(t)], [sp.sin(t), sp.cos(t)]])
Q0 = sp.Matrix([[1, 0], [0, sp.I]])
def qwp_Y(t): return sp.simplify(U(t).T * Q0 * U(t))

Q_fwd_Y, Q_bwd_Y = qwp_Y(sp.pi/4), qwp_Y(-sp.pi/4)
print("Q_forward same as Q above:", zero(Q_fwd_Y - Q))
show("Q_{backward}", Q_bwd_Y)

e_alpha_Y = sp.Matrix([sp.cos(alpha), sp.sin(alpha)])
E_out_Y   = sp.simplify(Q_bwd_Y * R_model * Q_fwd_Y * x_hat)
A_Y       = sp.simplify((e_alpha_Y.T * E_out_Y)[0])
I_Y       = sp.trigsimp(sp.simplify(sp.expand_complex(sp.expand(A_Y*sp.conjugate(A_Y)))))
show("I_{Yen}(\\alpha)", I_Y)

# compare with the rebuilt-convention result
I_Y_func = sp.lambdify((alpha, a, A, delta, phi), I_Y, 'numpy')
par = dict(a=1, A=0.4, delta=np.pi/3, phi=np.pi/3)
print("max |I_Yen(alpha) - I(-alpha)| =", np.max(np.abs(I_Y_func(ang, **par) - I_func(-ang, **par))))
plt.figure(figsize=(8, 4.5))
plt.plot(np.rad2deg(ang), I_func(ang, **par), label="ver1_rebuilt convention")
plt.plot(np.rad2deg(ang), I_Y_func(ang, **par), "--", label="Yen-san convention")
plt.xlabel(r"analyzer angle $\alpha$ (deg)"); plt.ylabel(r"$I(\alpha)$  (units of $a^2$)")
plt.xlim(0, 360); plt.grid(True); plt.legend()
plt.title(r"Same tensor ($a=1,A=0.4,\delta=\phi=60^\circ$), two conventions"); plt.tight_layout(); plt.show()

## Summary
With $R=\begin{pmatrix}a&Ae^{i\delta}\\-Ae^{i\delta}&ae^{i\phi}\end{pmatrix}$ and $\sigma_+$ incident:

$$R_{++}=\frac{a(1+e^{i\phi})}{2}+iAe^{i\delta},\qquad R_{-+}=\frac{a(1-e^{i\phi})}{2},$$

$$I(\alpha)=|R_{++}|^2\cos^2\alpha+|R_{-+}|^2\sin^2\alpha+2\,\mathrm{Im}(R_{++}R_{-+}^*)\sin\alpha\cos\alpha,$$

with $|R_{++}|^2=a^2\cos^2\frac\phi2+A^2+2aA\cos\frac\phi2\sin(\frac\phi2-\delta)$, $|R_{-+}|^2=a^2\sin^2\frac\phi2$, $\mathrm{Im}(R_{++}R_{-+}^*)=\frac{a^2}2\sin\phi+aA\sin\frac\phi2\sin(\frac\phi2-\delta)$.

* $\phi=0$ recovers the old result $I=(a^2+A^2-2aA\sin\delta)\cos^2\alpha$.
* $\phi\neq0$ opens a $\sigma_+\to\sigma_-$ channel and a $\sin\alpha\cos\alpha$ interference term, which tilts and reshapes the polar pattern.